# Multi-Factor Models
## 🎯 Learning Objectives

By the end of today you will be able to:

1. **Run and read a multi-factor regression** — CAPM, FF3, FF5, FF6
2. **Explain why alpha changes when you add a factor**, in both directions
3. **Argue that alpha is a property of a model, not of a strategy**
4. **Distinguish the time-series and cross-sectional approaches** to estimating
   the same thing
5. **Run a Fama-MacBeth regression** and interpret its slopes as portfolio returns

## 📋 Today's Plan

1. [From one factor to many](#many)
2. [Pitfall checklist](#pitfalls)
3. [🔄 Live Demo: the alpha ladder](#demo)
4. [Alpha is relative to a model — and ARKK vs Berkshire again](#relative)
5. [Two ways to estimate the same thing](#twoways)
6. [Fama-MacBeth](#fm)
7. [🛠️ Hands-On: run your signal up the ladder](#ho1)
8. [🎯 Challenge: momentum](#challenge) — *homework*
9. [Key takeaways](#takeaways)

---

## 🛠️ Setup

In [ ]:
#@title Setup — run this first
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = [11, 4.5]
import warnings; warnings.filterwarnings('ignore')
import pandas_datareader.data as web

BASE  = "https://raw.githubusercontent.com/amoreira2/UG54/refs/heads/main/assets/data"
panel = pd.read_parquet(f"{BASE}/panel_backbone_1980_2000.parquet")

f5  = web.DataReader('F-F_Research_Data_5_Factors_2x3','famafrench',start='1980-01-01')[0]/100
umd = web.DataReader('F-F_Momentum_Factor','famafrench',start='1980-01-01')[0]/100
for x in (f5, umd):
    x.index = pd.to_datetime(x.index.to_timestamp()) + pd.offsets.MonthEnd(0)
umd.columns = ['UMD']
FF = f5.join(umd, how='inner').loc['1980-01-31':'2000-12-31']

print(f"{len(FF)} months, factors: {[c for c in FF.columns if c != 'RF']}")

---

## 1. From One Factor to Many <a id="many"></a>

In Lecture 4 you regressed on the market alone:

$$r^e_t = \alpha + \beta \, r^e_{m,t} + \varepsilon_t$$

That model says the only thing you can buy cheaply is market exposure. Lecture 5
said otherwise: there are whole families of characteristics that have paid, and
you can buy any of them in an ETF for a few basis points.

So the honest model has more terms:

$$r^e_t = \alpha + \sum_{k} \beta_k f_{k,t} + \varepsilon_t$$

### The standard ladder

| Model | Factors | Source |
|---|---|---|
| **CAPM** | Mkt-RF | Sharpe (1964) |
| **FF3** | + SMB (size), HML (value) | Fama-French (1993) |
| **FF5** | + RMW (profitability), CMA (investment) | Fama-French (2015) |
| **FF6** | + UMD (momentum) | Carhart (1997) |

Each factor is a **long-short portfolio** — exactly the thing you built in
Lecture 3. HML is long high book-to-market, short low. That is why the ladder
matters: every factor added is a bet somebody is already selling cheaply, and
your α has to be something *else*.

> **📌 Remember what α means now**
>
> α is the average return your strategy earned that **none of the factors in the
> model** can explain. Change the model, change the α. It is not a fixed
> property of your strategy.

---

## 🛡️ Pitfall Checklist for Multi-Factor Regressions <a id="pitfalls"></a>

| | Pitfall | What goes wrong | 🔍 How to detect |
|---|---|---|---|
| 1 | **Reporting α without saying which model** | "Our alpha is 6%" is meaningless alone | Always write "α relative to FF3" |
| 2 | **Adding factors until α disappears** | You can kill any α with enough factors — that's not a finding | Did you pick the model *before* looking? |
| 3 | **Correlated factors** | HML and CMA overlap; loadings get unstable and hard to read | Check the factor correlation matrix |
| 4 | **Comparing α across different samples** | FF5 starts later than FF3 in some datasets | Are all models fitted on identical months? |
| 5 | **Reading a loading as a claim about holdings** | β on HML = 1.2 doesn't mean you hold HML | It means your returns *co-move* with it |
| 6 | **Ignoring the t-stat on α while celebrating its size** | Big α on 60 months is noise | |t| > 2, and how many months? |

> **🤖 AI-Era Insight**
>
> Pitfall 2 is the subtle one. Ask an AI to "test whether this strategy has
> alpha" and it will pick a model for you — usually FF3 — without telling you
> that the choice determines the answer. You'll see today that the same strategy
> has α of +18% or +3% depending only on which model you asked for.

---

## 🔄 Live Demo: The Alpha Ladder <a id="demo"></a>

Take the value long-short from Lecture 3 and run it up the ladder.

> **📝 Spec**
>
> Build the BM long-short (NYSE breakpoints, value-weighted, D10−D1, lagged,
> shifted to the month earned). Regress it on CAPM, then FF3, then FF5, then
> FF6, all on the same months, each with an intercept. Report annualized α, its
> t-stat, R², and the loadings.

> **🤔 Predict.** As you add factors, does α go up, down, or stay put?

In [ ]:
panel['me_l1'] = panel.groupby('permno')['me'].shift(1)   # Lecture 2 convention

def long_short(sig):
    s = pd.read_parquet(f"{BASE}/signals/{sig}.parquet")
    d = panel.merge(s, on=['permno','date'], how='left').sort_values(['permno','date'])
    d['sig_l1'] = d.groupby('permno')[sig].shift(1)
    d = d.dropna(subset=['sig_l1', 'ret', 'me_l1'])
    q = (d[d.exchcd == 1].groupby('date')['sig_l1'].quantile([.1,.9]).unstack()
           .rename(columns={0.1:'lo', 0.9:'hi'}))
    d = d.merge(q, on='date')
    d['g'] = np.where(d.sig_l1 <= d.lo, 0, np.where(d.sig_l1 >= d.hi, 9, np.nan))
    d = d.dropna(subset=['g'])
    p = d.groupby(['date','g']).apply(lambda g: np.average(g['ret'], weights=g['me_l1'])).unstack()
    return (p[9] - p[0]).dropna()      # already dated by the month earned

MODELS = {'CAPM': ['Mkt-RF'],
          'FF3' : ['Mkt-RF','SMB','HML'],
          'FF5' : ['Mkt-RF','SMB','HML','RMW','CMA'],
          'FF6' : ['Mkt-RF','SMB','HML','RMW','CMA','UMD']}

def ladder(sig, show=True):
    r = long_short(sig)
    j = pd.concat([r.rename('y'), FF], axis=1).dropna()
    out = {}
    if show:
        print(f"{sig} long-short — raw return {j.y.mean()*12:+.2%}/yr, "
              f"{len(j)} months\n")
        print(f"{'model':7s}{'alpha/yr':>11s}{'t':>7s}{'R²':>7s}   loadings")
        print("-"*72)
    for name, cols in MODELS.items():
        m = sm.OLS(j.y, sm.add_constant(j[cols])).fit()
        out[name] = m
        if show:
            ld = "  ".join(f"{c}={m.params[c]:+.2f}" for c in cols)
            print(f"{name:7s}{m.params['const']*12:>10.2%}{m.tvalues['const']:>7.2f}"
                  f"{m.rsquared:>7.2f}   {ld}")
    return out

_ = ladder('BM')

### The alpha didn't shrink. It vanished.

CAPM says value earned **+7.15%/yr with t = 2.45** — a real anomaly. FF3 says
**−1.16%, t = −0.70** — nothing at all.

Look at the loading that did it: **HML = 1.21**, and R² jumps from 0.06 to 0.71.

That is not surprising once you see it. HML *is* a value long-short — Fama and
French built it by sorting on book-to-market, which is exactly what we did. We
have rediscovered HML and then asked whether it beats HML.

> **💡 Key Insight**
>
> The regression is not saying value doesn't work. It is saying value doesn't
> work **beyond what you could already buy in a value ETF**. Those are different
> claims, and only the second one justifies a fee.

---

## 2. Alpha Is Relative to a Model <a id="relative"></a>

If adding factors always killed alpha, this would be a simple story. It doesn't.
Here are three more signals up the same ladder.

In [ ]:
for s in ['GP', 'Mom12m', 'NOA']:
    _ = ladder(s); print()

### Four signals, four completely different stories

| | CAPM → FF6 | What happened |
|---|---|---|
| **BM** | +7.2% → −0.7% | **Dies.** HML loading 1.21 — it *is* the factor |
| **GP** | +6.3% → **+9.3%** at FF3 → +5.9% at FF5 | **Grows, then shrinks** |
| **Mom12m** | +18.2% → +22.4% at FF5 → **+3.5%** at FF6 | **Survives five factors, dies at the sixth** |
| **NOA** | +11.8% → +6.8%, t stays above 2.4 throughout | **Survives everything** |

**Why GP's alpha grows.** Profitable firms tend to be *expensive* — GP loads
**−0.43** on HML. So under FF3 the model expects GP to lose money on its value
exposure. It didn't, so the unexplained part gets *bigger*. Controlling for a
factor you're negatively exposed to makes you look better, not worse.

**Why momentum dies only at FF6.** UMD loading is **1.46** and R² jumps from
0.12 to 0.84. UMD is a momentum long-short. Same story as BM and HML, one rung
later on the ladder.

> **💡 Key Insight: "does this strategy have alpha?" is not a well-posed question**
>
> Momentum has an α of +18% or +3% depending only on whether the person asking
> includes UMD. Neither number is wrong. **α is a property of the pair
> (strategy, model)** — you cannot report one without the other.
>
> This is the same lesson as the information ratio in Lecture 4, in regression
> form. There the benchmark was a choice; here the model is a choice. Both
> determine the answer.

> **⚠️ Caution: which model should *you* use?**
>
> The defensible rule is to fix the model **before** you look, and to justify it
> by what an investor could actually buy cheaply. If a low-cost momentum ETF
> exists, UMD belongs in your model, and momentum's α is 3.5% not 18%.
>
> The indefensible version is running all four and reporting whichever is
> biggest. That is pitfall 2, and it is depressingly common.

### ARKK and Berkshire, again

In *Performance Evaluation* you took Berkshire apart against the market alone and
got β = 0.59, R² = 0.22, an appraisal ratio of 0.38. You are about to get 0.63,
0.22 and 0.40 instead — from the same fund, over the same years.

**The file changed, not the fund.** The monthly series used in Lecture 4 spans
398 months but contains only 279 of them: scattered single months are missing,
three or four a year, all the way through. Nobody chose those months — it is a
merge artifact — but a third of the sample was quietly gone. This lecture uses a
version rebuilt from the daily file, with all 397 months present. Berkshire
barely moves. ARKK, which had 59 of its 86 months, moves a lot.

Keep that in view while you read the table. The first thing that decides an
alpha is not the model — it is whether you looked at your data.


In [ ]:
# Rebuilt from the daily file: all 397 months, no gaps. Funds compounded from
# daily returns, factors from Ken French's monthly files.
# Built by chapters/Finance/build_funds_monthly.py
funds = pd.read_pickle(f"{BASE}/df_WarrenBAndCathieW_monthly.pkl")
funds.columns = [c.strip() for c in funds.columns]      # 'Mom   ' has trailing spaces
SIX = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'Mom']

# The point of this section is that the old file was missing months, so say so
# loudly rather than quietly reporting the old numbers if it loads anyway.
assert len(funds) == 397, f"expected the rebuilt 397-month file, got {len(funds)}"


for nm in ['ARKK', 'BRK']:
    d = funds.dropna(subset=[nm])
    y = d[nm] - d['RF']
    print(f"{nm}, {len(y)} months")
    print(f"  {'model':14s}{'alpha':>8s}{'t':>7s}{'R²':>6s}{'appraisal':>11s}")
    for label, F in [('market only', ['Mkt-RF']), ('six factors', SIX)]:
        m  = sm.OLS(y, sm.add_constant(d[F])).fit()
        iv = m.resid.std() * np.sqrt(12)
        print(f"  {label:14s}{m.params['const']*12:>8.1%}{m.tvalues['const']:>7.2f}"
              f"{m.rsquared:>6.2f}{m.params['const']*12/iv:>11.2f}")
    print("  six-factor loadings: " + "  ".join(f"{f} {m.params[f]:+.2f}" for f in SIX) + "\n")

### Read the loadings before the alpha

**ARKK** loads `Mkt-RF +1.49`, `HML −0.95`, `RMW −0.89`, `SMB +0.60`: levered,
growth, unprofitable, small. The six factors explain 80% of its variance, up from
57%. Its alpha barely moves — 4.5% to 4.3% — but its residual risk shrinks, so
its appraisal ratio **rises**, from 0.23 to 0.32, on t = 0.76 over 86 months.

**Berkshire** loads `Mkt-RF +0.76`, `HML +0.51`, `RMW +0.25`: defensive, value,
profitable. Part of what looked like alpha against the market was that style, so
its alpha falls from 7.3% to 5.3% a year and its appraisal ratio from 0.40 to
0.32.

The formula from *Performance Evaluation* carries over, with the best mix of the
factors in place of the market:

$$SR_{\max} = \sqrt{SR_F^2 + AR^2}$$

So the appraisal ratio against six factors is what a fund adds for someone who
already holds all six.

> **💡 Key Insight: the model decides the ranking**
>
> Against the market, Berkshire looks clearly better than ARKK — 0.40 against
> 0.23. Against six factors the gap closes completely: **0.32 each**. Berkshire's
> edge over ARKK *was* its style. Value and quality can now be bought as an ETF,
> so against a model that contains them, that edge stops counting as alpha.
>
> Note which way each one moved. Adding factors took Berkshire's number down and
> pushed ARKK's up. "More factors is a tougher test" is not a rule.

⚠️ Neither t-statistic clears 2. On this evidence you cannot reject zero alpha for
either fund against the six-factor model. The ranking above is a statement about
point estimates, and the point estimates are not distinguishable from each other
or from nothing.


### The same fit at the daily frequency

Nothing above says the model has to be estimated monthly. The daily file carries
the **daily** versions of the same six factors next to the two funds, so the
regression is identical — only `12` becomes `252`.

Daily data buys 21× the observations. The question is what that actually buys you.


In [ ]:
fundsD = pd.read_pickle('https://raw.githubusercontent.com/amoreira2/Fin418/'
                        'main/assets/data/df_WarrenBAndCathieW.pkl')
fundsD.columns = [c.strip() for c in fundsD.columns]   # same trailing-space fix

for nm in ['ARKK', 'BRK']:
    d = fundsD.dropna(subset=[nm])
    y = d[nm] - d['RF']
    print(f"{nm}, {len(y)} days  ({d.index[0]:%Y-%m} to {d.index[-1]:%Y-%m})")
    print(f"  {'model':14s}{'alpha':>8s}{'t':>7s}{'R²':>6s}{'appraisal':>11s}")
    for label, F in [('market only', ['Mkt-RF']), ('six factors', SIX)]:
        m  = sm.OLS(y, sm.add_constant(d[F])).fit()
        iv = m.resid.std() * np.sqrt(252)                      # 252, not 12
        print(f"  {label:14s}{m.params['const']*252:>8.1%}{m.tvalues['const']:>7.2f}"
              f"{m.rsquared:>6.2f}{m.params['const']*252/iv:>11.2f}")
    print("  six-factor loadings: " + "  ".join(f"{f} {m.params[f]:+.2f}" for f in SIX) + "\n")

# Daily vs monthly on the *identical* sample: compound the same daily rows up to
# months, so the only thing changing is the horizon, not the window or the data.
# Note this compounds the daily FACTORS too, which is deliberate here and is not
# the same as French's monthly factor files -- he rebuilds those from monthly
# returns, and for Mom the two differ by ~70bp in the average month. For a
# genuine monthly study use his monthly files; for a clean horizon comparison
# you want one underlying dataset viewed at two frequencies, as here.
print(f"{'six factors, same sample':26s}{'alpha':>8s}{'t':>7s}{'R²':>6s}{'appr':>7s}   "
      + "".join(f"{f:>8s}" for f in SIX))
for nm in ['ARKK', 'BRK']:
    d = fundsD.dropna(subset=[nm])
    for label, dd, ppy in [('daily', d, 252),
                           ('monthly', (1 + d).resample('ME').prod() - 1, 12)]:
        m  = sm.OLS(dd[nm] - dd['RF'], sm.add_constant(dd[SIX])).fit()
        iv = m.resid.std() * np.sqrt(ppy)
        print(f"  {nm + ' ' + label:24s}{m.params['const']*ppy:>8.1%}{m.tvalues['const']:>7.2f}"
              f"{m.rsquared:>6.2f}{m.params['const']*ppy/iv:>7.2f}   "
              + "".join(f"{m.params[f]:>8.2f}" for f in SIX))

### What the daily data changes — and what it doesn't

**The betas move.** Berkshire's `RMW` goes from **−0.17 daily to +0.24 monthly**;
its `SMB` from −0.10 to −0.41, its `Mkt-RF` from 0.63 to 0.76. ARKK's market beta
goes from 1.17 to 1.44. Same fund, same window, same factors — only the horizon
is different.

This is not a bug. A daily beta measures how much of the factor move a fund picks
up *the same day*. Exposure that arrives with a lag — because the fund's stocks
or the factor's legs trade less often, or because the fund's positions reprice
slowly — shows up at the monthly horizon and not at the daily one. Daily betas
are biased toward zero for exactly that piece. **Report the horizon along with
the model.**

**The t-statistic does not improve.** This is the part that surprises people.
The t on alpha is

$$t(\alpha) \;\approx\; AR \times \sqrt{\text{years of data}}$$

— the appraisal ratio times the square root of the *span*, not the square root of
the number of observations. Berkshire's 8,338 days and its 398 months cover the
same 33 years, so they give you about the same precision on alpha. Sampling more
often within a year tells you almost nothing new about the mean.

> **💡 Key Insight: frequency helps betas, not alphas**
>
> Going daily shrinks the standard errors on the **loadings** a lot — you are
> measuring co-movement, and there is 21× more co-movement to see. It does
> essentially nothing for the standard error on **alpha**, which is a statement
> about an average return and needs calendar time. The only cure for a noisy
> alpha is a longer sample.

⚠️ One caution before you reach for daily data reflexively: the appraisal ratios
here differ across horizons (Berkshire 0.40 daily vs 0.30 monthly) *because the
betas differ*, not because the fund is better at one horizon. Pick the horizon
that matches how the strategy is actually held, and stay with it.


---

## 3. Two Ways to Estimate the Same Thing <a id="twoways"></a>

Everything so far ran **one regression per strategy, through time**:

$$r_{p,t} = \alpha + \beta' f_t + \varepsilon_t \qquad t = 1 \dots T$$

That is the **time-series** approach. It needs the factor returns $f_t$ to
already exist — someone had to build HML before you could regress on it.

There is a second way. Run **one regression per month, across stocks**:

$$r_{i,t+1} = \gamma_{0,t} + \gamma_t' x_{i,t} + e_{i,t+1} \qquad i = 1 \dots N$$

Here $x_{i,t}$ is stock *i*'s characteristic — its book-to-market, its size —
and the *slope* $\gamma_t$ is estimated fresh every month.

| | Time-series | Cross-sectional |
|---|---|---|
| One regression per | strategy | month |
| You must supply | factor **returns** | firm **characteristics** |
| You estimate | betas, α | the factor return itself |
| Runs on | ~250 months | ~250 × 4,000 firm-months |

> **💡 Key Insight: the slope IS a portfolio return**
>
> Not "behaves like." **Is.** Write the month-*t* regression in matrix form with
> characteristics $X$ and returns $r$, and OLS gives
> $\gamma_t = (X'X)^{-1}X'r$ — which is a fixed matrix times the vector of
> returns. That is the definition of a portfolio return, and $(X'X)^{-1}X'$ is
> the matrix of portfolio weights. You do not have to take this on faith; it
> falls out of the formula.
>
> So $\gamma_t$ is the return, in month *t*, of a portfolio with one unit of
> exposure to that characteristic and zero exposure to the others. It is a
> **pure play** — the closest thing to "what did value pay this month, holding
> size and profitability fixed."
>
> A decile sort can't do that. Sorting on book-to-market also sorts, partly, on
> size. A cross-sectional regression controls for the others by construction.

This is **Fama-MacBeth** (1973), and it is one of the most-used procedures in
empirical finance.

---

## 4. Fama-MacBeth <a id="fm"></a>

The procedure is two steps and no more:

1. **Every month**, regress next month's returns across stocks on this month's
   characteristics. Keep the slopes.
2. **Average the slopes over time.** The t-statistic comes from the *time series*
   of monthly slopes — not from any single regression.

> **🤔 Before we run it — why not just do it in one go?**
>
> You have a million stock-months. Stack them all and run **one** regression of
> `ret` on the lagged characteristics. Same coefficients, far simpler, and a
> million observations instead of 251.
>
> What is wrong with that? Say what step 2 is protecting you from.


In [ ]:
# === YOUR TURN ===
MY_PROMPT = """
                                    ← write your prompt here
"""

# ---- paste the AI's code below ----


In [ ]:
#@title 🔒 Reference implementation — later cells use `d`, `Z` and `G`
d = panel.dropna(subset=['ret', 'me_l1']).copy()
for s in ['BM','GP','Mom12m']:
    d = d.merge(pd.read_parquet(f"{BASE}/signals/{s}.parquet"), on=['permno','date'], how='left')
d = d.sort_values(['permno','date'])

# Every characteristic is lagged one month; the return is this month's (Lecture 2)
d['logME'] = np.log(d['me_l1'])
for s in ['BM','GP','Mom12m']:
    d[s] = d.groupby('permno')[s].shift(1)

CH = ['logME','BM','GP','Mom12m']
d  = d.dropna(subset=CH + ['ret'])

# winsorize + z-score each month so the slopes are comparable across characteristics
for c in CH:
    d[c+'_z'] = d.groupby('date')[c].transform(
        lambda v: (v.clip(v.quantile(.01), v.quantile(.99)) - v.mean()) / v.std())
Z = [c+'_z' for c in CH]

# STEP 1 — one cross-sectional regression per month
slopes = []
for _, g in d.groupby('date'):
    if len(g) < 100: continue
    slopes.append(sm.OLS(g['ret'], sm.add_constant(g[Z])).fit().params)
G = pd.DataFrame(slopes)

print(f"Step 1: {len(G)} monthly cross-sectional regressions, "
      f"~{len(d)/d.date.nunique():.0f} stocks each\n")

# STEP 2 — average the slopes; t-stat from their time-series variation
print(f"{'characteristic':16s}{'slope/month':>13s}{'annualized':>12s}{'t-stat':>9s}")
print("-"*50)
for c in ['const'] + Z:
    v = G[c]
    print(f"{('intercept' if c=='const' else c[:-2]):16s}"
          f"{v.mean():>13.5f}{v.mean()*12:>11.2%}{v.mean()/v.std()*np.sqrt(len(v)):>9.2f}")

In [ ]:
#@title 🔒 Check — run after the Fama-MacBeth cell above
# Same characteristics, same sample. Fama-MacBeth vs one pooled regression.
pooled = sm.OLS(d['ret'], sm.add_constant(d[Z])).fit()

print(f"{'characteristic':14s}{'slope/yr':>11s}{'FM t':>9s}{'pooled t':>11s}{'inflation':>12s}")
print("-"*57)
for c in Z:
    v = G[c]
    ft = v.mean()/v.std()*np.sqrt(len(v))
    pt = pooled.tvalues[c]
    print(f"{c[:-2]:14s}{v.mean()*12:>10.2%}{ft:>9.2f}{pt:>11.2f}{abs(pt/ft):>11.1f}x")
print(f"\n  pooled uses {len(d):,} observations; Fama-MacBeth uses {len(G)} monthly slopes.")

### Three to six times too confident

The slopes barely move. The **t-statistics inflate by 3–6×**, and one of them
changes the answer: size goes from **t = −1.81**, which does not clear the bar,
to **t = −11.69**, which would be among the most certain facts in finance.

The pooled regression believes it has a million independent observations. It
does not. Within any single month, stocks move together — that is the entire
premise of this course — so those thousands of rows carry far less information
than their count suggests. Averaging the *monthly* slopes and taking the
standard error across months prices that in, because months really are close to
independent.

> **📌 Remember**
>
> Step 2 is not a formality. It is the only thing standing between you and a
> t-statistic six times larger than the data supports.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(G.index if not isinstance(G.index, pd.RangeIndex) else range(len(G)),
        G['BM_z'].values, linewidth=0.9, label='BM slope')
ax.axhline(G['BM_z'].mean(), color='crimson', ls='--',
           label=f"mean {G['BM_z'].mean()*12:+.1%}/yr")
ax.axhline(0, color='black', lw=0.8)
ax.set_ylabel('monthly slope'); ax.set_title(
    'The value slope, month by month — the t-stat comes from THIS variation',
    fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()

print(f"months the value slope was positive: {(G['BM_z']>0).mean():.0%}")

### Reading the output

Each row is the return to a portfolio that is **+1 standard deviation** of that
characteristic and **flat on the other three**.

- **BM: +5.55%/yr, t = 4.99.** Value pays, controlling for size, profitability
  and momentum.
- **Mom12m: +5.56%/yr, t = 3.68.** So does momentum.
- **GP: +2.77%/yr, t = 3.79.** Smaller, but very reliable.
- **logME: −3.16%/yr, t = −1.81.** Bigger firms earn less — the size effect,
  and it doesn't clear |t| > 2 in this sample. Consistent with Lecture 3.

> **📌 Remember: the intercept is not alpha**
>
> The 14.96% intercept is the return of a stock with *average* values of every
> characteristic. It is roughly the market return, not a mispricing.

> **📎 What you do with it next**
>
> The time-series model gave you a *hedge*: hold the asset, short β units of the
> factor, and the systematic part cancels (Lecture 4). The cross-sectional model
> has an exact counterpart — compute the return your portfolio's
> **characteristics** imply, and subtract it. That's a **characteristic-adjusted
> return**, and it's hedging without ever estimating a time-series beta.
>
> It needs the multi-asset machinery, so it's Lecture 7.

> **🤔 Compare with the sort**
>
> In Lecture 3, the BM long-short under the standard
> convention (NYSE breakpoints, value-weighted) gave +5.27%/yr. Fama-MacBeth gives +5.55%/yr
> — close, but not identical, and the FM version *holds size, profitability and
> momentum fixed* while the sort does not. When they disagree sharply, it means
> your sort was picking up something other than the characteristic you sorted
> on.

---

## 🛠️ Hands-On: Run Your Signal Up the Ladder <a id="ho1"></a>

You have followed one signal since Lecture 3. Time to find out what it really
has.

> **🤔 Predict.** Given what you learned in Lecture 5 about your signal's
> nearest neighbours — does it load on one of the FF factors? Which one?

In [ ]:
# === EDIT + YOUR TURN ===
MY_SIGNAL = "GP"       # ← your pick

res = ____             # hint: ladder(MY_SIGNAL)   — it prints the table for you

a_capm = res['CAPM'].params['const']*12
a_ff6  = res['FF6'].params['const']*12
print(f"\n{MY_SIGNAL}: alpha falls from {a_capm:+.2%} (CAPM) to {a_ff6:+.2%} (FF6)")
print(f"  that is {1 - a_ff6/a_capm:.0%} of the CAPM alpha explained by the other five factors")
print(f"  biggest loading in FF6: "
      f"{res['FF6'].params.drop('const').abs().idxmax()}")

### What to say out loud

- **Which factor took the biggest bite?** If it's HML your signal is a value
  play; if UMD, a momentum play; if RMW, profitability.
- **Did your α survive FF6 with |t| > 2?** If yes, you have something the
  standard model can't explain — which is genuinely interesting and also the
  moment to get suspicious. Lectures 8 and 9.
- **Did your α get bigger?** Then you're negatively exposed to a factor that
  paid, and the raw return understated you.

---

## 🎯 Challenge: Momentum <a id="challenge"></a>

*Homework — due before Lecture 7.*

Momentum is the most profitable signal in our whole menu — **+19.9%/yr** raw.
Your PM wants to know whether to pay for a momentum manager.

Run `Mom12m` up the full ladder and report.

### Q1 — The alpha ladder

> **📌 Required variable names:**
> ```python
> mom_alpha_capm = ____   # annualized alpha vs CAPM
> mom_alpha_ff3  = ____
> mom_alpha_ff5  = ____
> mom_alpha_ff6  = ____
> ```

In [ ]:
# Your work here


# Required outputs — fill these in:
mom_alpha_capm = ____
mom_alpha_ff3  = ____
mom_alpha_ff5  = ____
mom_alpha_ff6  = ____

for n, a in [('CAPM',mom_alpha_capm),('FF3',mom_alpha_ff3),
             ('FF5',mom_alpha_ff5),('FF6',mom_alpha_ff6)]:
    print(f"  {n:5s} alpha {a:+7.2%}/yr")

### Q2 — What did it

Report momentum's UMD loading and R² under FF6.

> **📌 Required variable names:**
> ```python
> mom_umd_beta = ____   # loading on UMD in the FF6 regression
> mom_r2_ff6   = ____   # R-squared of the FF6 regression
> ```

In [ ]:
# Your work here


# Required outputs — fill these in:
mom_umd_beta = ____
mom_r2_ff6   = ____

print(f"UMD loading {mom_umd_beta:+.2f}   R² {mom_r2_ff6:.2f}")

### Q3 — The memo

> **📝 Your task — maximum 6 sentences**
>
> Should your PM pay for a momentum manager?
>
> The α is X, Y, Z, ... against CAPM, ff3, ff5, FF6. Say which number you'd put
> in front of the investment committee and defend it. Explain what the UMD
> loading  means in plain language. And say what would have to be true
> about the market for the CAPM number to be the right one.

In [ ]:
MEMO = """
Write your memo here. Don't delete the surrounding triple quotes.
"""
print(MEMO)

---

## 📤 Submission <a id="submit"></a>

In [ ]:
# === 📤 SUBMISSION CELL — Run this last ===
import json, base64, hashlib, datetime as dt

required = ["mom_alpha_capm", "mom_alpha_ff3", "mom_alpha_ff5",
            "mom_alpha_ff6", "mom_umd_beta", "mom_r2_ff6", "MEMO"]
missing = [v for v in required if v not in globals()]
if missing:
    raise NameError(f"\n❌ Missing before submission: {missing}")

payload = {
    "assignment": "L6_MultiFactor_AI",
    "ts": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z"),
    "answers": {k: float(eval(k)) for k in required if k != "MEMO"},
    "memo": MEMO.strip(),
}
blob = json.dumps(payload, sort_keys=True)
checksum = hashlib.sha256(blob.encode()).hexdigest()[:8]
token = f"UG54::{checksum}::{base64.b64encode(blob.encode()).decode()}"

print("=" * 72)
print("📋  COPY THE LINE BELOW AND PASTE INTO THE SUBMISSION FORM")
print("=" * 72)
print(token)
print("=" * 72)
print("Submission form: https://forms.gle/yazZ8bbatL87jdJi7")

---

## 🧠 Key Takeaways <a id="takeaways"></a>

1. **Every factor in the model is a long-short portfolio** — the thing you built
   in Lecture 3. Adding one asks "can you beat *that* too?"

2. **α is a property of (strategy, model), not of a strategy.** Momentum's α is
   +18% or +3.5% depending only on whether UMD is in the model.

3. **Never report an alpha without naming the model.** The same goes for the
   appraisal ratio: against six factors ARKK's rises from 0.23 to 0.32 and
   Berkshire's falls from 0.40 to 0.32 — a clear ranking against the market
   becomes a tie.

4. **Alpha can grow when you add a factor.** GP's α rises under FF3 because it
   loads *negatively* on HML.

5. **Name the horizon too.** Estimating the same six-factor model daily instead
   of monthly moves Berkshire's RMW loading from +0.24 to −0.17. Higher frequency
   sharpens the betas and does nothing for the alpha's t-statistic, which needs
   calendar span, not observations.

6. **Check the file before you trust the number.** The monthly series used in
   Lecture 4 was missing a third of its months. Berkshire's appraisal ratio moved
   from 0.38 to 0.40 once they were restored; ARKK's six-factor number moved from
   0.63 to 0.32.

7. **Fix the model before you look.** Justify it by what an investor could
   actually buy cheaply. Running all four and reporting the best is not analysis.

8. **Two estimation routes to the same object.** Time-series needs factor
   returns and gives you betas; cross-sectional needs characteristics and gives
   you the factor return.

9. **A Fama-MacBeth slope is a portfolio return** — a pure play on one
   characteristic, holding the others fixed. That's something a sort cannot do.

10. **The FM t-stat comes from variation across months**, not from the thousands
   of stocks inside any single month.

---

### Next class

Everything so far has been in-sample. We have run the same data through four
models and picked the interesting results. Next: what that does to a t-statistic,
and how you would ever know whether any of it is real.

---

## 📎 Appendix <a id="appendix"></a>

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 📎 APPENDIX — Belt-and-Suspenders Data Loading
# ═══════════════════════════════════════════════════════════════════════
#   panel = pd.read_parquet(f"{BASE}/panel_backbone_1980_2000.parquet")
#
# Factors come live from Ken French. Prompt:
# "Using pandas-datareader fetch F-F_Research_Data_5_Factors_2x3 and
#  F-F_Momentum_Factor, monthly, from 1980. Convert PeriodIndex to month-end,
#  divide by 100, and join them."
def fetch_ff6():
    import pandas_datareader.data as web
    f5  = web.DataReader('F-F_Research_Data_5_Factors_2x3','famafrench',start='1980-01-01')[0]/100
    umd = web.DataReader('F-F_Momentum_Factor','famafrench',start='1980-01-01')[0]/100
    for x in (f5, umd):
        x.index = pd.to_datetime(x.index.to_timestamp()) + pd.offsets.MonthEnd(0)
    umd.columns = ['UMD']
    return f5.join(umd, how='inner')

# Note: Ken French labels the momentum factor 'Mom   ' with trailing spaces in
# some releases. We rename to 'UMD' on load rather than trusting the label.
